In [1]:
import wandb
wandb.login()

/Users/gurpuramit/Desktop/MLOps/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/gurpuramit/.netrc.


wandb: Currently logged in as: gurpuramit (gurpuramit-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import wandb
import numpy as np
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.metrics import precision_recall_fscore_support

run = wandb.init(project="Lab1-visualize-models", name="xgboost")

# Load dermatology dataset from OpenML (more reliable than direct UCI download)
dermatology = fetch_openml(name='dermatology', version=1, as_frame=False, parser='liac-arff')
X = dermatology.data.astype(float)
# Labels are 1-6; shift to 0-5 for XGBoost
Y = dermatology.target.astype(int) - 1

split = int(len(X) * 0.7)
train_X, test_X = X[:split], X[split:]
train_Y, test_Y = Y[:split], Y[split:]

xg_train = xgb.DMatrix(train_X, label=train_Y)
xg_test  = xgb.DMatrix(test_X,  label=test_Y)

# Setup parameters for xgboost
param = {}
param['objective'] = 'multi:softmax'
param['eta'] = 0.1
param['max_depth'] = 6
param['silent'] = 1
param['nthread'] = 4
param['num_class'] = 6
wandb.config.update(param)

watchlist = [(xg_train, 'train'), (xg_test, 'test')]
num_round = 5

bst = xgb.train(param, xg_train, num_round, watchlist, callbacks=[wandb.xgboost.WandbCallback()])

# --- Feature 1: Feature Importance Bar Chart ---
importance_scores = bst.get_score(importance_type='gain')
sorted_importance = sorted(importance_scores.items(), key=lambda x: x[1], reverse=True)
feat_names = [item[0] for item in sorted_importance]
feat_scores = [item[1] for item in sorted_importance]
importance_table = wandb.Table(data=[[n, s] for n, s in zip(feat_names, feat_scores)],
                               columns=["Feature", "Importance (Gain)"])
wandb.log({"feature_importance": wandb.plot.bar(importance_table, "Feature", "Importance (Gain)",
                                                 title="Feature Importance (Gain)")})

# Get predictions
pred = bst.predict(xg_test)
error_rate = np.sum(pred != test_Y) / test_Y.shape[0]
print('Test error using softmax = {}'.format(error_rate))
run.summary['Error Rate'] = error_rate

# --- Feature 2: Per-Class Metrics Table ---
class_names = ['psoriasis', 'seboreic dermatitis', 'lichen planus',
               'pityriasis rosea', 'chronic dermatitis', 'pityriasis rubra pilaris']
precision, recall, f1, support = precision_recall_fscore_support(test_Y, pred, labels=[0, 1, 2, 3, 4, 5])
metrics_table = wandb.Table(columns=["Class", "Precision", "Recall", "F1-Score", "Support"])
for i, name in enumerate(class_names):
    metrics_table.add_data(name, round(precision[i], 4), round(recall[i], 4),
                           round(f1[i], 4), int(support[i]))
wandb.log({"per_class_metrics": metrics_table})
run.summary['macro_f1'] = round(f1.mean(), 4)
run.summary['macro_precision'] = round(precision.mean(), 4)
run.summary['macro_recall'] = round(recall.mean(), 4)

wandb.sklearn.plot_confusion_matrix(test_Y, pred, [0., 1., 2., 3., 4., 5.])

run.finish()

wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /Users/gurpuramit/Desktop/MLOps/Labs/Experiment_Tracking_Labs/W&B/wandb/run-20260411_012704-tdpvm8oi
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run xgboost


wandb: ⭐️ View project at https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models


wandb: 🚀 View run at https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models/runs/tdpvm8oi


...

wandb: uploading artifact run-tdpvm8oi-feature_importance_table; uploading artifact run-tdpvm8oi-per_class_metrics; uploading artifact run-tdpvm8oi-confusion_matrix


wandb: uploading wandb-summary.json; uploading media/table/Feature Importance_table_5_7252382006239c3f7329.table.json; uploading media/table/feature_importance_table_6_e4580d0239a2faf2dd30.table.json; uploading media/table/per_class_metrics_7_4abe5f9da6d54652bdf9.table.json; uploading output.log (+ 4 more)


wandb: 
wandb: Run history:
wandb:          epoch ▁▃▅▆█
wandb:  test-mlogloss █▆▄▂▁
wandb: train-mlogloss █▆▄▂▁
wandb: 
wandb: Run summary:
wandb:      Error Rate 0.08182
wandb:           epoch 4
wandb:        macro_f1 0.9007
wandb: macro_precision 0.9065
wandb:    macro_recall 0.9355
wandb: 


wandb: 🚀 View run xgboost at: https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models/runs/tdpvm8oi
wandb: ⭐️ View project at: https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models
wandb: Synced 5 W&B file(s), 4 media file(s), 8 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260411_012704-tdpvm8oi/logs


In [3]:
# --- Feature 3: Hyperparameter Sweep ---
# Searches over eta, max_depth, and num_round to find the best combination
from sklearn.metrics import f1_score
from sklearn.datasets import fetch_openml

# Pre-load data once so each sweep trial doesn't re-fetch
_dermatology = fetch_openml(name='dermatology', version=1, as_frame=False, parser='liac-arff')
_X = _dermatology.data.astype(float)
_Y = _dermatology.target.astype(int) - 1
_split = int(len(_X) * 0.7)

def train_sweep():
    with wandb.init() as run:
        cfg = run.config

        train_X, test_X = _X[:_split], _X[_split:]
        train_Y, test_Y = _Y[:_split], _Y[_split:]
        xg_train = xgb.DMatrix(train_X, label=train_Y)
        xg_test  = xgb.DMatrix(test_X,  label=test_Y)

        param = {
            'objective': 'multi:softmax',
            'eta': cfg.eta,
            'max_depth': cfg.max_depth,
            'silent': 1,
            'nthread': 4,
            'num_class': 6
        }

        bst = xgb.train(param, xg_train, cfg.num_round,
                        [(xg_train, 'train'), (xg_test, 'test')],
                        callbacks=[wandb.xgboost.WandbCallback()])

        pred = bst.predict(xg_test)
        error_rate = np.sum(pred != test_Y) / test_Y.shape[0]
        macro_f1 = f1_score(test_Y, pred, average='macro')
        run.summary['error_rate'] = error_rate
        run.summary['macro_f1'] = macro_f1

sweep_config = {
    'method': 'random',
    'metric': {'name': 'macro_f1', 'goal': 'maximize'},
    'parameters': {
        'eta':      {'values': [0.05, 0.1, 0.3]},
        'max_depth':{'values': [4, 6, 8]},
        'num_round':{'values': [5, 15, 30]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="Lab1-visualize-models")
wandb.agent(sweep_id, train_sweep, count=9)

Create sweep with ID: ywch0ehh
Sweep URL: https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models/sweeps/ywch0ehh


wandb: Agent Starting Run: jbvzfidm with config:


wandb: 	eta: 0.05


wandb: 	max_depth: 4


wandb: 	num_round: 5


...

wandb: uploading artifact run-q66gwwt8-FeatureImportance_table-4m962g


wandb: uploading history steps 0-15, summary, console lines 2-16


wandb: 
wandb: Run history:
wandb:          epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb:  test-mlogloss █▇▆▅▄▄▃▃▃▂▂▂▁▁▁
wandb: train-mlogloss █▇▆▅▄▄▃▃▃▂▂▂▁▁▁
wandb: 
wandb: Run summary:
wandb:      epoch 14
wandb: error_rate 0.08182
wandb:   macro_f1 0.9007
wandb: 


wandb: 🚀 View run usual-sweep-9 at: https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models/runs/q66gwwt8
wandb: ⭐️ View project at: https://wandb.ai/gurpuramit-northeastern-university/Lab1-visualize-models
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260411_012756-q66gwwt8/logs
